# Workshop 3 · 00 — Setup (bereitgestellt)

Dieses Notebook erzeugt die **Ausgangsdaten** für alle folgenden Aufgaben. Es enthält bewusst uneinheitliche und unvollständige Daten:
- Stammdaten teils leer / uneinheitlich (`baureihe`, `depot`), `komponente` fehlt
- `freitext_meldung` (Werkstatt-Freitext — die Struktur steckt nur im Text)
- `kunden_kommentar` (Material für Sentiment & Keywords)
- `foto_pfad` (Verweis auf Wagon-Fotos für den CV-Teaser)

> **Aufgabe:** Hier gibt es nichts zu implementieren — führe einfach beide Code-Zellen aus, um Tabelle `asset_meldungen` und die Fotos anzulegen.

**Voraussetzung:** Ein Lakehouse ist an dieses Notebook angehängt.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType
import random
from datetime import date, timedelta

random.seed(7)

# Absichtlich uneinheitliche Stammdaten
baureihen = ['ICE 4', 'ice4', 'ICE-4', 'ICE3', 'ic 2', None, 'Talent 2', 'FLIRT 3', None, 'Doppelstock']
depots = ['HH-Eidelstedt', 'Hamburg Eidelstedt', 'hamburg-eidelstedt', 'FFM Griesheim',
          'Frankfurt-Griesheim', 'Muenchen Pasing', 'muenchen pasing', 'Leipzig Hbf', 'Berlin Rummelsburg']

# Im Freitext steckt die Wahrheit: Komponente, Ort, Schweregrad
freitexte = [
    'Tuer 2 klemmt beim Schliessen, leichte Roststelle am Tuerrahmen, nicht sicherheitskritisch',
    'Riss in der Seitenwand Wagen 3, dringend pruefen, sicherheitsrelevant',
    'Graffiti grossflaechig an der Aussenwand, rein kosmetisch',
    'Klimaanlage Abteil 5 ohne Funktion, mehrere Beschwerden',
    'Delle am Drehgestell nach Steinschlag, Strukturpruefung empfohlen',
    'WC im Wagen 1 defekt, Geruchsbelaestigung, mittlere Prioritaet',
    'Fenster laesst sich nicht verriegeln, Sicherheitsrisiko bei Fahrt',
    'Sitzpolster Reihe 12 eingerissen, kosmetisch',
    'Bremsbelag stark abgenutzt, sofort in die Werkstatt',
    'Beleuchtung Eingangsbereich flackert, geringe Prioritaet',
]
kommentare = [
    'Puenktlich und sauber, sehr zufrieden!',
    'WLAN ging die ganze Fahrt nicht, aergerlich.',
    'Klimaanlage viel zu kalt, unangenehm.',
    'Freundliches Personal, gerne wieder.',
    'Zug 20 Minuten zu spaet, Anschluss verpasst.',
    'Sitze bequem, aber sehr laut im Wagen.',
    'Toilette war defekt und verschmutzt.',
    'Top Reise, alles bestens.',
]

# Explizites Schema -> komponente/foto_pfad sind teils None
schema = StructType([
    StructField('meldung_id',       StringType(), False),
    StructField('wagon_id',         StringType(), False),
    StructField('baureihe',         StringType(), True),
    StructField('depot',            StringType(), True),
    StructField('komponente',       StringType(), True),
    StructField('freitext_meldung', StringType(), True),
    StructField('kunden_kommentar', StringType(), True),
    StructField('foto_pfad',        StringType(), True),
    StructField('eingang',          StringType(), True),
])

rows = []
for i in range(24):
    foto = ('Files/wagon_photos_w3/W3-%02d.png' % i) if i < 6 else None
    rows.append(Row(
        meldung_id='M-%03d' % i,
        wagon_id='W3-%02d' % i,
        baureihe=random.choice(baureihen),
        depot=random.choice(depots),
        komponente=None,
        freitext_meldung=freitexte[i % len(freitexte)],
        kunden_kommentar=kommentare[i % len(kommentare)],
        foto_pfad=foto,
        eingang=(date(2026, 1, 1) + timedelta(days=random.randint(0, 160))).isoformat(),
    ))

sdf = spark.createDataFrame(rows, schema=schema)
sdf.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('asset_meldungen')
print('Zeilen:', sdf.count())
display(sdf)

## Wagon-Fotos bereitstellen
Lädt 6 Beispiel-Fotos aus einem öffentlichen GitHub-Repo in `Files/wagon_photos_w3` (für den CV-Teaser in Notebook 03). Einfach ausführen.

In [ ]:
import os, io, zipfile, random, urllib.request, urllib.error
from PIL import Image, ImageDraw

OUT = '/lakehouse/default/Files/wagon_photos_w3'
os.makedirs(OUT, exist_ok=True)
random.seed(7)

GH_OWNER  = 'henrik-ra'
GH_REPO   = 'fabric-ai-workshop-images'
GH_BRANCH = 'main'
GH_FOLDER = 'wagon'

UA = {'User-Agent': 'fabric-ai-workshop/1.0 (workshop demo)'}
ZIP_URL = 'https://github.com/%s/%s/archive/refs/heads/%s.zip' % (GH_OWNER, GH_REPO, GH_BRANCH)
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

geladen = set()
try:
    req = urllib.request.Request(ZIP_URL, headers=UA)
    with urllib.request.urlopen(req, timeout=60) as r:
        blob = r.read()
    zf = zipfile.ZipFile(io.BytesIO(blob))
    marker = ('/%s/' % GH_FOLDER) if GH_FOLDER else '/'
    treffer = sorted(
        n for n in zf.namelist()
        if not n.endswith('/') and marker in n and n.lower().endswith(EXTS)
    )
    for i, name in enumerate(treffer[:6]):
        img = Image.open(io.BytesIO(zf.read(name))).convert('RGB')
        img.save(OUT + '/W3-%02d.png' % i)
        geladen.add(i)
        print('geladen:', name.split('/')[-1])
    if not treffer:
        print('Repo erreichbar, aber keine Bilder in %s gefunden' % GH_FOLDER)
except Exception as e:
    print('Repo-Download fehlgeschlagen (%s)' % type(e).__name__)

print('Fotos bereit in', OUT, '| echt:', len(geladen))